In [6]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import rdFMCS
from rdkit.Chem import rdMolAlign
from collections import defaultdict
import subprocess
import itertools
import copy


csv_path = "CSV/Riboflavin-smiles-combined3.xlsx"
df = pd.read_excel(csv_path)
smiles_list = df['Smiles'].tolist()
print(smiles_list)

mols = []
for i, smi in enumerate(smiles_list):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        print(f"Could not parse SMILES {smi}")
        continue
    mol = Chem.AddHs(mol)  # add hydrogens explicitly
    AllChem.EmbedMolecule(mol, AllChem.ETKDG())  # generate 3D conformer
    AllChem.UFFOptimizeMolecule(mol)  # optimize geometry
    mols.append((f"mol_{i}", mol))

mol_dir = "mol_files"
os.makedirs(mol_dir, exist_ok=True)

for name, mol in mols:
    mol_block = Chem.MolToMolBlock(mol)  # This outputs the mol format string
    filename = os.path.join(mol_dir, f"{name}.mol")
    with open(filename, "w") as f:
        f.write(mol_block)
    print(f"Saved {filename}")

mol2_dir = "mol2_files"
os.makedirs(mol2_dir, exist_ok=True)
for filename in os.listdir(mol_dir):
    if filename.endswith(".mol"):
        mol_path = os.path.join(mol_dir, filename)
        mol2_path = os.path.join(mol2_dir, filename.replace(".mol", ".mol2"))
        try:
            subprocess.run(['obabel', mol_path, '-O', mol2_path], check=True)
            print(f"Converted {mol_path} -> {mol2_path}")
        except subprocess.CalledProcessError as e:
            print(f"Conversion failed for {mol_path}: {e}")

print("\n--- Identifying Variable Sites (MCS) ---")
raw_mols = [Chem.MolFromSmiles(smi) for smi in smiles_list if Chem.MolFromSmiles(smi)]

# Find Maximum Common Substructure (MCS)
mcs_result = rdFMCS.FindMCS(raw_mols)
mcs_smarts = mcs_result.smartsString
print(f"MCS SMARTS: {mcs_smarts}")

mcs_mol = Chem.MolFromSmarts(mcs_smarts)

# For each molecule, identify which atoms are not part of the MCS
variable_sites = []
for i, mol in enumerate(raw_mols):
    match = mol.GetSubstructMatch(mcs_mol)
    unmatched = [atom.GetIdx() for atom in mol.GetAtoms() if atom.GetIdx() not in match]
    variable_sites.append(unmatched)
    print(f"Molecule {i} -> Variable atom indices (non-MCS): {unmatched}")

# --- Step 3: Cluster variable atoms based on 3D position ---
print("\n--- Clustering Variable Atoms ---")

# First, align all molecules to Mol0 as reference (for spatial comparison)
# Align all molecules to Mol0 using MCS atom mapping
ref_mol = mols[0][1]  # only the mol object
ref_match = ref_mol.GetSubstructMatch(mcs_mol)
aligned_mols = []

for i, (name, mol) in enumerate(mols):
    mol_match = mol.GetSubstructMatch(mcs_mol)
    mol_copy = Chem.Mol(mol)  # clone to avoid in-place modification

    if mol_match and ref_match:
        # Create atom map between this mol and ref mol
        atom_map = list(zip(mol_match, ref_match))
        try:
            rdMolAlign.AlignMol(mol_copy, ref_mol, atomMap=atom_map)
            aligned_mols.append((name, mol_copy))
        except Exception as e:
            print(f"Alignment failed for {name}: {e}")
            aligned_mols.append((name, mol_copy))  # unaligned fallback
    else:
        print(f"Skipping alignment for {name}: no MCS match")
        aligned_mols.append((name, mol_copy))  # unaligned fallback

# Now cluster by position (we assume atoms in similar positions are same group)
cluster_tolerance = 1.5  # angstroms
clusters = []  # list of sets of (mol_index, atom_index)

# For each variable atom in each mol, try to group it
for i, (_, mol) in enumerate(aligned_mols):
    conf = mol.GetConformer()
    for atom_idx in variable_sites[i]:
        pos_i = conf.GetAtomPosition(atom_idx)

        # Try to assign this atom to an existing cluster
        assigned = False
        for cluster in clusters:
            for j, a_idx in cluster:
                other_conf = aligned_mols[j][1].GetConformer()
                pos_j = other_conf.GetAtomPosition(a_idx)
                dist = pos_i.Distance(pos_j)
                if dist < cluster_tolerance:
                    cluster.add((i, atom_idx))
                    assigned = True
                    break
            if assigned:
                break

        # If not close to any existing cluster, make a new one
        if not assigned:
            clusters.append(set([(i, atom_idx)]))

# Print clustered variable atoms
for c_idx, cluster in enumerate(clusters):
    print(f"Cluster {c_idx}: {sorted(cluster)}")

print("\n--- Generating all combinations of protonation states ---")
num_clusters = len(clusters)
print(f"Number of variable sites (clusters): {num_clusters}")

# Generate all combinations of 0/1 states for each cluster (2^num_clusters)

combinations = list(itertools.product([0, 1], repeat=num_clusters))
print(f"Total combinations to generate: {len(combinations)}")

# Directory for output variants
variants_dir = "mol2_variants"
os.makedirs(variants_dir, exist_ok=True)

for comb_idx, combo in enumerate(combinations):
    print(f"\nCombination {comb_idx}: {combo}")

    # For each molecule, create a modified copy with this protonation pattern
    for mol_idx, (name, orig_mol) in enumerate(mols):
        mol_copy = Chem.RWMol(copy.deepcopy(orig_mol))

        conf = mol_copy.GetConformer()

        # For each cluster/site
        for cluster_idx, cluster_state in enumerate(combo):
            # If cluster_state is 1, protonated (or presence), else deprotonated (or absence)
            # Here, just toggle hydrogens on atoms in this cluster accordingly

            cluster_atoms = [atom_idx for (m_idx, atom_idx) in clusters[cluster_idx] if m_idx == mol_idx]

            for atom_idx in cluster_atoms:
                atom = mol_copy.GetAtomWithIdx(atom_idx)
                # Simple example: add/remove explicit hydrogen attached to this atom
                # (This is just an example - you should customize logic)
                if cluster_state == 0:
                    # Remove hydrogens attached to this atom
                    neighbors = [nbr for nbr in atom.GetNeighbors() if nbr.GetAtomicNum() == 1]
                    for h in neighbors:
                        mol_copy.RemoveAtom(h.GetIdx())
                else:
                    # Add a hydrogen if not present (this is tricky — simplified here)
                    # If no hydrogen neighbor, add one
                    has_h = any(nbr.GetAtomicNum() == 1 for nbr in atom.GetNeighbors())
                    if not has_h:
                        new_h = Chem.Atom(1)
                        mol_copy.AddAtom(new_h)
                        mol_copy.AddBond(atom_idx, mol_copy.GetNumAtoms()-1, order=Chem.rdchem.BondType.SINGLE)

        # Save modified molecule variant to mol2 file
        variant_name = f"{name}_variant_{comb_idx}"
        variant_mol_path = os.path.join(variants_dir, f"{variant_name}.mol")

        mol_block = Chem.MolToMolBlock(mol_copy)
        with open(variant_mol_path, "w") as f:
            f.write(mol_block)

        # Convert to mol2 via Open Babel
        variant_mol2_path = variant_mol_path.replace(".mol", ".mol2")
        try:
            subprocess.run(['obabel', variant_mol_path, '-O', variant_mol2_path], check=True)
            print(f"Saved variant mol2: {variant_mol2_path}")
        except subprocess.CalledProcessError as e:
            print(f"Conversion failed for {variant_mol_path}: {e}")




['Cc1cc2nc3c(=O)[nH]c(=O)nc-3n(C[C@H](O)[C@H](O)[C@H](O)CO)c2cc1C ', 'Cc1cc2nc3c(=O)[nH]c(=O)nc-3n(C[C@H]([O-])[C@H](O)[C@H](O)CO)c2cc1C ', 'Cc1cc2nc3c([O-])nc(=O)nc-3n(C[C@H](O)[C@H](O)[C@H](O)CO)c2cc1C', 'Cc1cc2nc3c(=O)nc(O)nc-3n(C[C@H](O)[C@H](O)[C@H](O)CO)c2cc1C', 'Cc1cc2nc3c([O-])nc(=O)nc-3n(C[C@H]([O-])[C@H](O)[C@H](O)CO)c2cc1C ', 'Cc1cc2nc3c([O-])nc(=O)[nH+]c-3n(C[C@H](O)[C@H](O)[C@H](O)CO)c2cc1C']
Saved mol_files/mol_0.mol
Saved mol_files/mol_1.mol
Saved mol_files/mol_2.mol
Saved mol_files/mol_3.mol
Saved mol_files/mol_4.mol
Saved mol_files/mol_5.mol
Converted mol_files/mol_4.mol -> mol2_files/mol_4.mol2
Converted mol_files/mol_3.mol -> mol2_files/mol_3.mol2
Converted mol_files/mol_2.mol -> mol2_files/mol_2.mol2
Converted mol_files/mol_5.mol -> mol2_files/mol_5.mol2


1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted


Converted mol_files/mol_1.mol -> mol2_files/mol_1.mol2
Converted mol_files/mol_0.mol -> mol2_files/mol_0.mol2

--- Identifying Variable Sites (MCS) ---


1 molecule converted


MCS SMARTS: [#6]-[#6]1:[#6]:[#6]2:[#7]:[#6]3:[#6]:[#7]:[#6]:[#7]:[#6]-3:[#7](:[#6]:2:[#6]:[#6]:1-[#6])-[#6]-[#6](-[#8])-[#6](-[#8])-[#6](-[#8])-[#6]-[#8]
Molecule 0 -> Variable atom indices (non-MCS): [7, 10]
Molecule 1 -> Variable atom indices (non-MCS): [7, 10]
Molecule 2 -> Variable atom indices (non-MCS): [7, 10]
Molecule 3 -> Variable atom indices (non-MCS): [7, 10]
Molecule 4 -> Variable atom indices (non-MCS): [7, 10]
Molecule 5 -> Variable atom indices (non-MCS): [7, 10]

--- Clustering Variable Atoms ---
Cluster 0: [(0, 7), (4, 7), (5, 7)]
Cluster 1: [(0, 10), (1, 10), (2, 10), (3, 10), (4, 10), (5, 10)]
Cluster 2: [(1, 7), (2, 7), (3, 7)]

--- Generating all combinations of protonation states ---
Number of variable sites (clusters): 3
Total combinations to generate: 8

Combination 0: (0, 0, 0)
Saved variant mol2: mol2_variants/mol_0_variant_0.mol2
Saved variant mol2: mol2_variants/mol_1_variant_0.mol2
Saved variant mol2: mol2_variants/mol_2_variant_0.mol2
Saved variant mol2: 

...

1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted


Saved variant mol2: mol2_variants/mol_3_variant_6.mol2
Saved variant mol2: mol2_variants/mol_4_variant_6.mol2
Saved variant mol2: mol2_variants/mol_5_variant_6.mol2

Combination 7: (1, 1, 1)
Saved variant mol2: mol2_variants/mol_0_variant_7.mol2
Saved variant mol2: mol2_variants/mol_1_variant_7.mol2


1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted


Saved variant mol2: mol2_variants/mol_2_variant_7.mol2
Saved variant mol2: mol2_variants/mol_3_variant_7.mol2
Saved variant mol2: mol2_variants/mol_4_variant_7.mol2
Saved variant mol2: mol2_variants/mol_5_variant_7.mol2


1 molecule converted
1 molecule converted
1 molecule converted
